# 01 — Exploratory data analysis

Inspect the fixed raw dataset before cleaning. This notebook does not write to or alter the raw CSV. Correlation describes association, not causation.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'messy_fraud_dataset_20k.csv').exists(): ROOT = ROOT.parent
raw = pd.read_csv(ROOT / 'messy_fraud_dataset_20k.csv', low_memory=False)
raw.shape

In [ ]:
raw.head()

In [ ]:
raw.dtypes.to_frame('dtype').join(raw.isna().sum().rename('missing')).join(raw.nunique(dropna=False).rename('unique'))

In [ ]:
raw.duplicated().sum(), raw.describe(include='all').T

In [ ]:
target = 'is_fraud'
raw[target].value_counts(dropna=False).rename('count').to_frame().assign(percent=lambda x: x['count']/len(raw)*100)

In [ ]:
cat = raw.select_dtypes(include=['object','category']).columns
{c: raw[c].value_counts(dropna=False).head(30) for c in cat if raw[c].nunique(dropna=False) < 100}

In [ ]:
num = raw.select_dtypes(include='number').columns
raw[num].describe(percentiles=[.01,.25,.5,.75,.95,.99]).T

In [ ]:
sns.countplot(data=raw, x='txn_type', hue=target)
plt.xticks(rotation=35, ha='right'); plt.tight_layout()

In [ ]:
sns.histplot(data=raw, x='amount', hue=target, bins=60, element='step', stat='density', common_norm=False)

In [ ]:
timestamps = pd.to_datetime(raw.timestamp, format='%Y-%m-%d %H:%M:%S', errors='coerce')
mask = timestamps.isna()
timestamps.loc[mask] = pd.to_datetime(raw.loc[mask, 'timestamp'], format='%d/%m/%Y %H:%M', errors='coerce')
timestamps.isna().sum(), timestamps.min(), timestamps.max()

In [ ]:
sns.heatmap(raw[num].corr(), cmap='vlag', center=0); plt.title('Numeric feature correlations (association only)')

In [ ]:
# Outlier review uses quantiles; extreme values are flagged for inspection, not automatically deleted.
raw[num].quantile([.001,.01,.99,.999]).T